In [0]:
 Databricks notebook source
# ============================================================================
# STOCK MARKET ANALYTICS: Data Ingestion
# ============================================================================
# Purpose: Ingest real stock market data from Yahoo Finance
# Data Source: yfinance (https://finance.yahoo.com via API)
# Author: RSangDev
# Date: 2026-06-16
# ============================================================================
 
# MAGIC %md
# ## Stock Market Analytics - Data Ingestion
# Ingesting REAL stock price data from Yahoo Finance API
 

In [0]:

import yfinance as yf
import pandas as pd
import pyspark.sql.functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import time
 
catalog = "workspace"
schema = "stock_analytics"
 
print("=" * 70)
print("STOCK MARKET PIPELINE - DATA INGESTION")
print("=" * 70)

In [0]:

# Cell 1: Setup
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`{schema}`")
print(f"✅ Schema created: {catalog}.{schema}")
 
# Stocks to track (Popular, diverse sectors)
STOCKS = [
    "AAPL",      # Apple (Tech)
    "MSFT",      # Microsoft (Tech)
    "GOOGL",     # Google (Tech)
    "AMZN",      # Amazon (Retail/Cloud)
    "TSLA",      # Tesla (Auto/Energy)
    "META",      # Meta (Social)
    "NVDA",      # Nvidia (semiconductors)
    "JPM",       # JP Morgan (Finance)
    "JNJ",       # Johnson & Johnson (Healthcare)
    "PG",        # Procter & Gamble (Consumer)
]
 
print(f"📊 Tracking {len(STOCKS)} stocks: {', '.join(STOCKS)}")

In [0]:

# Cell 2: Fetch Stock Data from Yahoo Finance
print("\n" + "=" * 70)
print("FETCHING STOCK DATA FROM YAHOO FINANCE")
print("=" * 70 + "\n")
 
# Period: Last 5 years of historical data
all_stocks_data = []
start_date = (datetime.now() - timedelta(days=365*5)).strftime("%Y-%m-%d")
end_date = datetime.now().strftime("%Y-%m-%d")
 
print(f"Date range: {start_date} to {end_date}\n")
 
for i, stock in enumerate(STOCKS, 1):
    print(f"[{i}/{len(STOCKS)}] Fetching {stock}...", end=" ")
    try:
        # Fetch historical data from Yahoo Finance
        df = yf.download(stock, start=start_date, end=end_date, progress=False)
        
        if df is not None and len(df) > 0:
            # Reset index to make Date a column
            df.reset_index(inplace=True)
            
            # Add stock ticker
            df['Ticker'] = stock
            
            # Rename columns (lowercase for consistency)
            df.columns = ['Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume', 'Ticker']
            
            # Add metadata
            df['data_fetched_at'] = datetime.now().isoformat()
            df['source'] = 'yahoo_finance'
            
            all_stocks_data.append(df)
            print(f"✅ ({len(df)} records)")
        else:
            print("❌ No data")
            
    except Exception as e:
        print(f"❌ Error: {str(e)}")
    
    time.sleep(0.5)  # Be respectful to API
 
print(f"\n✅ Successfully fetched {len(all_stocks_data)} stocks")
 